# imports

In [5]:
import duckdb
import pandas as pd
import numpy as np
import os

# config

In [6]:
ASSET = 'BTC'
INTERVAL = '4h'
SEQUENCE_LENGTH = 20

db_path = '../../database/financial_data.duckdb'
conn = duckdb.connect(db_path)

print(f"Asset: {ASSET} | Interval: {INTERVAL} | Sequence: {SEQUENCE_LENGTH} candles")

Asset: BTC | Interval: 4h | Sequence: 20 candles


# Load raw OHLCV from Gold Layer (without indicators)

In [7]:
query = f"""
    SELECT date, open, high, low, close, volume
    FROM gold_crypto_features 
    WHERE asset_symbol = '{ASSET}' 
    AND interval = '{INTERVAL}' 
    ORDER BY date ASC
"""

print(f"load {ASSET} {INTERVAL} raw OHLCV from gold_crypto_features")
df = conn.execute(query).df()
conn.close()

print(f"Loaded: {df.shape}")
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
df.head(3)

load BTC 4h raw OHLCV from gold_crypto_features
Loaded: (13296, 6)
Date range: 2020-04-27 12:00:00 → 2026-05-22 08:00:00


,date,open,high,low,close,volume
0,2020-04-27 12:00:00,7713.5,7722.5,7624.0,7669.5,1068.971
1,2020-04-27 16:00:00,7669.5,7718.5,7642.0,7700.5,588.447
2,2020-04-27 20:00:00,7700.5,7777.5,7686.0,7772.5,838.893


# Compute log returns (stationary & scale-invariant)

Log returns: log(price_t / price_{t-1}) 
- Stationary (mean ≈ 0) → safe for LSTM
- Scale-invariant → BTC at $7k vs $77k treated the same

In [8]:
for col in ['open', 'high', 'low', 'close', 'volume']:
    df[f'{col}_logret'] = np.log(df[col] / df[col].shift(1))

df = df.dropna(subset=[f'{col}_logret' for col in ['open', 'high', 'low', 'close', 'volume']]).reset_index(drop=True)

logret_cols = ['open_logret', 'high_logret', 'low_logret', 'close_logret', 'volume_logret']

print(f"After log returns: {df.shape}")
print("\nLog return stats (should be mean≈0, no trend):")
df[logret_cols].describe().round(6)

After log returns: (13295, 11)

Log return stats (should be mean≈0, no trend):


,open_logret,high_logret,low_logret,close_logret,volume_logret
count,13295.000000,13295.000000,13295.000000,13295.000000,13295.000000
mean,0.000173,0.000173,0.000174,0.000174,0.000089
std,0.012412,0.011597,0.013737,0.012412,0.682790
min,-0.110651,-0.098760,-0.242322,-0.110651,-5.078599
25%,-0.004536,-0.004275,-0.004037,-0.004535,-0.460315
50%,0.000207,-0.000452,0.000856,0.000205,-0.044422
75%,0.004973,0.004334,0.005112,0.004973,0.424116
max,0.136807,0.156262,0.227640,0.136807,4.907352


# Create target (next-candle direction, same as XGBoost v4)

In [9]:
df['target_4h'] = df['close_logret'].shift(-1)
df['target_direction'] = (df['target_4h'] > 0).astype(int)

df = df.dropna(subset=['target_4h']).reset_index(drop=True)

print(f"After target creation: {df.shape}")
print(f"\nClass Balance:")
print(df['target_direction'].value_counts(normalize=True).mul(100).round(2))


After target creation: (13294, 13)

Class Balance:
target_direction
1    51.28
0    48.72
Name: proportion, dtype: float64
